In [7]:
print("WE LOVE FABRIC!!")

StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 9, Finished, Available, Finished, False)

WE LOVE FABRIC!!


#  FabCon Atlanta 2026 - From Data Analyst to Data Engineer
## Hands-On Demo Notebook: PySpark transformations + Engine optimisations

**Session:** Transition from Data Analyst to Data Engineer: Upgrading Skills from Dataflows Gen2 to Notebooks   
**Dataset:** E-commerce orders, customers, products (~1 400 rows across 5 files)

---

### 📋 What's in this notebook

| # | Topic | M / Power Query equivalent |
|---|-------|---------------------------|
| 0 | Setup & data load | Get Data -> CSV |
| 1 | Splitting columns | Split Column by Delimiter |
| 2 | Extract before / after delimiter | Extract Text Before/After Delimiter |
| 3 | Replacing values | Replace Values |
| 4 | Merging queries | Merge Queries |
| 5 | Appending queries | Append Queries |
| 6 | Pivoting | Pivot Column |
| 7 | Unpivoting | Unpivot Columns |
| 8 | Uppercase / Lowercase | Format -> UPPERCASE / lowercase |
| 9 | Trimming whitespace | Transform -> Trim |
| 10 | Incremental loading | Incremental Refresh (Dataflows Gen2) |
| 11 | Engine optimisations | *(not available in Dataflows)* |

> **Tip:** Run cells top-to-bottom. Each section is self-contained but builds on the Delta tables created in Section 0.


---
## 📦 Section 0 - Setup & Data Load

### What this does
Loads all five CSV files from the Lakehouse `Files/` layer into Spark DataFrames,
then writes them as **Delta tables** in the Lakehouse so every later section can
work directly against the managed table layer.

### M / Power Query equivalent
```
// In DataFlows Gen2: Data Sources -> Lakehouse -> Files -> CSV
// Repeat for each file; schema is auto-detected
```

> **Files expected in Lakehouse → Files/demo_data/**
> - `orders.csv` (1 000 rows)
> - `customers.csv` (300 rows)  
> - `products.csv` (94 rows)
> - `orders_new.csv` (200 rows - used in Section 10)
> - `sales_wide.csv` (5 rows - used in Section 7)


In [ ]:
# ── Step 0.1  Spark session is already available in Fabric Notebooks
# Just verify we can see the session
print(f"Spark version : {spark.version}")
print(f"Default DB    : {spark.catalog.currentDatabase()}")


In [8]:
# ── Step 0.2  Load CSVs from Lakehouse Files layer
# Change the base_path below if you uploaded files to a different folder

BASE = "Files/"

df_orders    = spark.read.csv(f"{BASE}/orders.csv",    header=True, inferSchema=True)
df_customers = spark.read.csv(f"{BASE}/customers.csv", header=True, inferSchema=True)
df_products  = spark.read.csv(f"{BASE}/products.csv",  header=True, inferSchema=True)
df_new       = spark.read.csv(f"{BASE}/orders_new.csv",header=True, inferSchema=True)
df_wide      = spark.read.csv(f"{BASE}/sales_wide.csv",header=True, inferSchema=True)

print(f"orders      : {df_orders.count():>5} rows  |  columns: {len(df_orders.columns)}")
print(f"customers   : {df_customers.count():>5} rows  |  columns: {len(df_customers.columns)}")
print(f"products    : {df_products.count():>5} rows  |  columns: {len(df_products.columns)}")
print(f"orders_new  : {df_new.count():>5} rows  |  columns: {len(df_new.columns)}")
print(f"sales_wide  : {df_wide.count():>5} rows  |  columns: {len(df_wide.columns)}")


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 10, Finished, Available, Finished, False)

orders      :  1000 rows  |  columns: 15
customers   :   300 rows  |  columns: 10
products    :    94 rows  |  columns: 9
orders_new  :   200 rows  |  columns: 15
sales_wide  :     5 rows  |  columns: 13


In [9]:
# ── Step 0.3  Quick schema peek — spot the dirty columns we'll clean later
print("=== orders schema ===")
df_orders.printSchema()

print("\n=== orders sample (first 3 rows) ===")
df_orders.select("order_id","product_info","shipping_info","status","amount").show(3, truncate=False)

print("\n=== customers sample ===")
df_customers.select("customer_id","full_name","email").show(3, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 11, Finished, Available, Finished, False)

=== orders schema ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_info: string (nullable = true)
 |-- shipping_info: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)


=== orders sample (first 3 rows) ===
+----------+--------------------------------------------+------------------------+---------+-------+
|order_id  |product_info                                |shipping_info           |status   |amount |
+----------+--------------------------------------------+------------------------+---------+-------+
|ORD-

In [10]:
# ── Step 0.4  Write base DataFrames as Delta tables
# This is our "bronze" layer — raw data, Delta format, no transformations yet

(df_orders
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_orders"))

(df_customers
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_customers"))

(df_products
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_products"))

print("Delta tables created: bronze_orders, bronze_customers, bronze_products")
spark.sql("SHOW TABLES").show(truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 12, Finished, Available, Finished, False)

Delta tables created: bronze_orders, bronze_customers, bronze_products
+---------------------------------+----------------+-----------+
|namespace                        |tableName       |isTemporary|
+---------------------------------+----------------+-----------+
|FabCon_Atlanta.DF_to_Notebook.dbo|bronze_customers|false      |
|FabCon_Atlanta.DF_to_Notebook.dbo|bronze_orders   |false      |
|FabCon_Atlanta.DF_to_Notebook.dbo|bronze_products |false      |
+---------------------------------+----------------+-----------+



---
## 🔪 Section 1 - Splitting Columns

### What this does
The `product_info` column stores three values in a single string, separated by a pipe character:

```
Electronics|Smartphones|iPhone 15 Pro
```

We need to **split** this into three separate columns: `category`, `subcategory`, and `product_name`.  
This is one of the most common transformations when ingesting data from APIs or flat files.

### M / Power Query equivalent
```
// Home -> Split Column -> By Delimiter
// Delimiter: |
// Split at: Each occurrence
// Output columns: product_info.1, product_info.2, product_info.3
// -> then rename each column manually
= Table.SplitColumn(
    Source,
    "product_info",
    Splitter.SplitTextByDelimiter("|", QuoteStyle.None),
    {"category", "subcategory", "product_name"}
)
```

> **Dataflows Gen2 limitation:** You must repeat this step for every query that has the same pattern.  
> **Notebook advantage:** Write a reusable helper function - one line of code per DataFrame.


In [11]:
from pyspark.sql.functions import split, col, trim

# ── Step 1.1  Inspect what we're working with
print("Sample product_info values:")
spark.read.table("bronze_orders") \
    .select("order_id", "product_info") \
    .distinct() \
    .show(8, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 13, Finished, Available, Finished, False)

Sample product_info values:
+----------+---------------------------------------------+
|order_id  |product_info                                 |
+----------+---------------------------------------------+
|ORD-000574|Books & Media|Non-Fiction|Future Minds       |
|ORD-000588|Books & Media|Comics|Galactic Wanderers Vol.1|
|ORD-000929|Home & Garden|Home Decor|LED Strip Lights 5m |
|ORD-000652|Home & Garden|Kitchen|Espresso Machine       |
|ORD-000426|Home & Garden|Garden|Raised Garden Bed 120cm |
|ORD-000169|Clothing|Kids' Apparel|Graphic T-Shirt Pack  |
|ORD-000419|Books & Media|Comics|Galactic Wanderers Vol.1|
|ORD-000656|Books & Media|Comics|Galactic Wanderers Vol.1|
+----------+---------------------------------------------+
only showing top 8 rows



In [12]:
# ── Step 1.2  Split product_info (Category|Subcategory|ProductName) into 3 columns
#
# split(col, pattern) -> returns an ArrayType column
# .getItem(n)         -> extracts element at index n (0-based); returns null if out of range
# r"\|"              -> escaped pipe in Java regex (the pipe | is a special character)

df = spark.read.table("bronze_orders")

parts = split(col("product_info"), r"\|")

df_split = (df
    .withColumn("category",     trim(parts.getItem(0)))
    .withColumn("subcategory",  trim(parts.getItem(1)))
    .withColumn("product_name", trim(parts.getItem(2)))
    .drop("product_info")        # drop the original combined column
)

df_split.select("order_id", "category", "subcategory", "product_name").show(10, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 14, Finished, Available, Finished, False)

+----------+-------------+-------------+-----------------------+
|order_id  |category     |subcategory  |product_name           |
+----------+-------------+-------------+-----------------------+
|ORD-000018|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000157|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000174|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000290|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000551|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000572|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000602|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000935|Home & Garden|Garden       |Solar Lantern Set 4pk  |
|ORD-000003|Home & Garden|Garden       |Raised Garden Bed 120cm|
|ORD-000004|Clothing     |Kids' Apparel|Waterproof Raincoat    |
+----------+-------------+-------------+-----------------------+
only showing top 10 rows



In [13]:
# ── Step 1.3  Reusable helper function
# The REAL advantage over Dataflows: define once, use everywhere

def split_pipe_column(df, source_col, new_col_names):
    """
    Split a pipe-delimited column into multiple new columns.
    Drops the original column after splitting.
    
    Args:
        df           : Input DataFrame
        source_col   : Name of the column to split (e.g. 'product_info')
        new_col_names: List of new column names for each split part
    
    Returns:
        DataFrame with original column replaced by split columns
    """
    parts = split(col(source_col), r"\|")
    for idx, name in enumerate(new_col_names):
        df = df.withColumn(name, trim(parts.getItem(idx)))
    return df.drop(source_col)


# Apply to orders — split product_info
df_orders_split = split_pipe_column(
    spark.read.table("bronze_orders"),
    source_col    = "product_info",
    new_col_names = ["category", "subcategory", "product_name"]
)

# Verify
print(f"Row count: {df_orders_split.count()}")
df_orders_split.select("order_id","category","subcategory","product_name").show(5, truncate=False)

# Save as silver layer table
(df_orders_split
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("silver_orders_s1"))

print("\n Saved as silver_orders_s1")


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 15, Finished, Available, Finished, False)

Row count: 1000
+----------+-------------+-----------+---------------------+
|order_id  |category     |subcategory|product_name         |
+----------+-------------+-----------+---------------------+
|ORD-000018|Home & Garden|Garden     |Solar Lantern Set 4pk|
|ORD-000157|Home & Garden|Garden     |Solar Lantern Set 4pk|
|ORD-000174|Home & Garden|Garden     |Solar Lantern Set 4pk|
|ORD-000290|Home & Garden|Garden     |Solar Lantern Set 4pk|
|ORD-000551|Home & Garden|Garden     |Solar Lantern Set 4pk|
+----------+-------------+-----------+---------------------+
only showing top 5 rows


 Saved as silver_orders_s1


---
## ✂️ Section 2 - Extract Before / After Delimiter

### What this does
The `shipping_info` column contains `Carrier|TrackingNumber` in a single string:

```
FedEx|TRK847392015
```

Sometimes you only need **one part** - just the carrier name, or just the tracking number.  
This is more targeted than a full split: extract everything **before** or **after** a delimiter.

### M / Power Query equivalent
```
// Extract text BEFORE the delimiter:
// Add Column -> Extract -> Text Before Delimiter -> "|"
= Table.AddColumn(Source, "carrier", each Text.BeforeDelimiter([shipping_info], "|"))

// Extract text AFTER the delimiter:
// Add Column -> Extract -> Text After Delimiter -> "|"
= Table.AddColumn(Source, "tracking_number", each Text.AfterDelimiter([shipping_info], "|"))
```


In [14]:
from pyspark.sql.functions import split, col, trim, regexp_extract

# ── Step 2.1  Quick look at the raw shipping_info values
spark.read.table("bronze_orders") \
    .select("order_id", "shipping_info") \
    .show(6, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 16, Finished, Available, Finished, False)

+----------+------------------------+
|order_id  |shipping_info           |
+----------+------------------------+
|ORD-000018|DHL|SHP527219856        |
|ORD-000157|TNT Express|TRK961609527|
|ORD-000174|UPS|PKG744053238        |
|ORD-000290|USPS|SHP748939011       |
|ORD-000551|DPD|SHP767774608        |
|ORD-000572|DHL|DEL317055967        |
+----------+------------------------+
only showing top 6 rows



In [15]:
# ── Step 2.2  Extract BEFORE delimiter — carrier name
#
# Two approaches — pick whichever is clearest for your team:

# Approach A: split + getItem (most readable, reuses the split pattern from Section 1)
from pyspark.sql.functions import split, col, trim

df = spark.read.table("bronze_orders")

df_carrier = df.withColumn(
    "carrier",
    trim(split(col("shipping_info"), r"\|").getItem(0))  # everything BEFORE the pipe
)

# Approach B: regexp_extract (single regex, no array intermediate)
from pyspark.sql.functions import regexp_extract

df_carrier_b = df.withColumn(
    "carrier",
    regexp_extract(col("shipping_info"), r"^([^|]+)", 1)  # capture group: chars before |
)

# Show both to confirm they're identical
print("Approach A (split+getItem):")
df_carrier.select("order_id","shipping_info","carrier").show(5, truncate=False)

print("Approach B (regexp_extract):")
df_carrier_b.select("order_id","shipping_info","carrier").show(5, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 17, Finished, Available, Finished, False)

Approach A (split+getItem):
+----------+------------------------+-----------+
|order_id  |shipping_info           |carrier    |
+----------+------------------------+-----------+
|ORD-000018|DHL|SHP527219856        |DHL        |
|ORD-000157|TNT Express|TRK961609527|TNT Express|
|ORD-000174|UPS|PKG744053238        |UPS        |
|ORD-000290|USPS|SHP748939011       |USPS       |
|ORD-000551|DPD|SHP767774608        |DPD        |
+----------+------------------------+-----------+
only showing top 5 rows

Approach B (regexp_extract):
+----------+------------------------+-----------+
|order_id  |shipping_info           |carrier    |
+----------+------------------------+-----------+
|ORD-000018|DHL|SHP527219856        |DHL        |
|ORD-000157|TNT Express|TRK961609527|TNT Express|
|ORD-000174|UPS|PKG744053238        |UPS        |
|ORD-000290|USPS|SHP748939011       |USPS       |
|ORD-000551|DPD|SHP767774608        |DPD        |
+----------+------------------------+-----------+
only showing top 5

In [16]:
# ── Step 2.3  Extract AFTER delimiter — tracking number

df_shipping_clean = (df
    .withColumn("carrier",         trim(split(col("shipping_info"), r"\|").getItem(0)))
    .withColumn("tracking_number", trim(split(col("shipping_info"), r"\|").getItem(1)))
    .drop("shipping_info")
)

df_shipping_clean.select("order_id","carrier","tracking_number").show(8, truncate=False)

# ── Step 2.4  Count orders per carrier (a quick sanity check)
print("\nOrders per carrier:")
df_shipping_clean.groupBy("carrier").count().orderBy("count", ascending=False).show()


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 18, Finished, Available, Finished, False)

+----------+-----------+---------------+
|order_id  |carrier    |tracking_number|
+----------+-----------+---------------+
|ORD-000018|DHL        |SHP527219856   |
|ORD-000157|TNT Express|TRK961609527   |
|ORD-000174|UPS        |PKG744053238   |
|ORD-000290|USPS       |SHP748939011   |
|ORD-000551|DPD        |SHP767774608   |
|ORD-000572|DHL        |DEL317055967   |
|ORD-000602|USPS       |PKG587718440   |
|ORD-000935|Evri       |TRK756169258   |
+----------+-----------+---------------+
only showing top 8 rows


Orders per carrier:
+-----------+-----+
|    carrier|count|
+-----------+-----+
|TNT Express|  114|
|        UPS|  111|
|       Evri|  107|
|        DHL|  104|
|     Hermes|  101|
|      FedEx|  100|
|       USPS|   95|
|        GLS|   90|
|        DPD|   89|
| Royal Mail|   89|
+-----------+-----+



In [ ]:
# ── Step 2.5  Generalised helper: extract_part()
#
# Wraps the split pattern into a clean function you can use across any column

def extract_part(df, source_col, delimiter, part, new_col_name):
    """
    Extract one part from a delimited column.
    
    Args:
        part : 'before' → index 0  |  'after' → index 1  |  int → explicit index
    """
    parts_array = split(col(source_col), f"\\{delimiter}" if delimiter in r"\.+*?^${}()|[]" else delimiter)
    idx = {"before": 0, "after": 1}.get(part, part)
    return df.withColumn(new_col_name, trim(parts_array.getItem(idx)))


# Demo: extract carrier (before) and tracking (after) in two clean calls
df_s2 = (spark.read.table("bronze_orders")
    .transform(lambda df: extract_part(df, "shipping_info", "|", "before", "carrier"))
    .transform(lambda df: extract_part(df, "shipping_info", "|", "after",  "tracking_number"))
    .drop("shipping_info")
)

df_s2.select("order_id","carrier","tracking_number").show(5, truncate=False)
print(f"\n Extracted carrier and tracking_number for {df_s2.count()} orders")


---
## 🔄 Section 3 - Replacing Values

### What this does
The `status` column in our orders data is a mess - the same value was entered in
multiple ways by different systems:

| Variants | Canonical value |
|----------|----------------|
| `Completed`, `completed`, `COMPLETED`, `Complete` | `Completed` |
| `Shipped`, `shipped`, `SHIPPED`, `Dispatched`, `dispatched` | `Shipped` |
| `Pending`, `pending`, `PENDING`, `In Progress` | `Pending` |
| `Cancelled`, `cancelled`, `CANCELLED`, `Canceled` | `Cancelled` |
| `Returned`, `returned` | `Returned` |

We'll standardise all variants to a single canonical value using three different
PySpark approaches - choose whichever fits your team's style best.

### M / Power Query equivalent
```
// Home -> Replace Values (one rule at a time)
= Table.ReplaceValue(Source, "completed", "Completed", Replacer.ReplaceText, {"status"})
// -> must repeat for every variant → creates a long chain of steps
```
> **Dataflows Gen2 limitation:** one replacement per step, no regex support, no bulk mapping.  
> **Notebook advantage:** handle all variants in a single cell.


In [17]:
# ── Step 3.1  Audit the dirty values
from pyspark.sql.functions import col, lower, count

print("All distinct status values (and how common each is):")
(spark.read.table("bronze_orders")
    .groupBy("status")
    .count()
    .orderBy("count", ascending=False)
    .show(30, truncate=False))


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 19, Finished, Available, Finished, False)

All distinct status values (and how common each is):
+-----------+-----+
|status     |count|
+-----------+-----+
|Completed  |177  |
|Shipped    |144  |
|Pending    |88   |
|completed  |69   |
|shipped    |61   |
|COMPLETED  |60   |
|Complete   |48   |
|pending    |43   |
|Cancelled  |42   |
|SHIPPED    |41   |
|cancelled  |37   |
|In Progress|36   |
|Returned   |31   |
|Dispatched |28   |
|PENDING    |27   |
|CANCELLED  |26   |
|returned   |21   |
|dispatched |13   |
|Canceled   |8    |
+-----------+-----+



In [ ]:
# ── Step 3.2  Approach A: regexp_replace  (case-insensitive regex)
#
# Best for: collapsing many text variants with a single pattern
# (?i) = case-insensitive flag in Java regex

from pyspark.sql.functions import regexp_replace

df = spark.read.table("bronze_orders")

df_regex = (df
    .withColumn("status",
        regexp_replace(col("status"), r"(?i)^(complete[d]?|completed)$", "Completed"))
    .withColumn("status",
        regexp_replace(col("status"), r"(?i)^(shipped|dispatched)$", "Shipped"))
    .withColumn("status",
        regexp_replace(col("status"), r"(?i)^(pending|in progress)$", "Pending"))
    .withColumn("status",
        regexp_replace(col("status"), r"(?i)^(cancel+ed?)$", "Cancelled"))
    .withColumn("status",
        regexp_replace(col("status"), r"(?i)^(returned?)$", "Returned"))
)

print("After regexp_replace — distinct values:")
df_regex.groupBy("status").count().orderBy("count", ascending=False).show()


In [18]:
# ── Step 3.3  Approach B: when / otherwise  (analyst-friendly, most readable)
#
# Best for: explicit mapping where each condition is visible in code

from pyspark.sql.functions import when, lower, trim

df_when = (df
    .withColumn("status_clean",
        when(lower(trim(col("status"))).isin("completed","complete"), "Completed")
        .when(lower(trim(col("status"))).isin("shipped","dispatched"),  "Shipped")
        .when(lower(trim(col("status"))).isin("pending","in progress"), "Pending")
        .when(lower(trim(col("status"))).isin("cancelled","canceled"),  "Cancelled")
        .when(lower(trim(col("status"))).isin("returned"),              "Returned")
        .otherwise(col("status"))   # keep anything unrecognised as-is
    )
)

print("Mapping preview (original → clean):")
df_when.select("status","status_clean").distinct().orderBy("status").show(30, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 20, Finished, Available, Finished, False)

Mapping preview (original → clean):
+-----------+------------+
|status     |status_clean|
+-----------+------------+
|CANCELLED  |Cancelled   |
|COMPLETED  |Completed   |
|Canceled   |Cancelled   |
|Cancelled  |Cancelled   |
|Complete   |Completed   |
|Completed  |Completed   |
|Dispatched |Shipped     |
|In Progress|Pending     |
|PENDING    |Pending     |
|Pending    |Pending     |
|Returned   |Returned    |
|SHIPPED    |Shipped     |
|Shipped    |Shipped     |
|cancelled  |Cancelled   |
|completed  |Completed   |
|dispatched |Shipped     |
|pending    |Pending     |
|returned   |Returned    |
|shipped    |Shipped     |
+-----------+------------+



In [ ]:
# ── Step 3.4  Approach C: mapping dictionary  (best for long lookup tables)
#
# Best for: large reference lists loaded from a config file or database table

from pyspark.sql.functions import create_map, lit
from itertools import chain

status_map = {
    "completed":   "Completed", "complete":    "Completed",
    "COMPLETED":   "Completed", "Complete":    "Completed",
    "shipped":     "Shipped",   "SHIPPED":     "Shipped",
    "dispatched":  "Shipped",   "Dispatched":  "Shipped",
    "pending":     "Pending",   "PENDING":     "Pending",
    "In Progress": "Pending",
    "cancelled":   "Cancelled", "CANCELLED":   "Cancelled",
    "Canceled":    "Cancelled",
    "returned":    "Returned",
}

# Build a Spark MapType column from the Python dict
mapping_expr = create_map([lit(x) for x in chain(*status_map.items())])

df_mapped = (df
    .withColumn("status",
        # coalesce: if key not in map, fall back to original value
        mapping_expr[col("status")].alias("status_mapped")
    )
)

# More Pythonic approach using DataFrame.replace():
df_replaced = df.replace(status_map, subset=["status"])

print("After .replace() — distinct values:")
df_replaced.groupBy("status").count().orderBy("count", ascending=False).show()


In [19]:
# ── Step 3.5  Save cleaned orders (using the when/otherwise approach — most readable)

df_status_clean = (spark.read.table("bronze_orders")
    .withColumn("status",
        when(lower(trim(col("status"))).isin("completed","complete"), "Completed")
        .when(lower(trim(col("status"))).isin("shipped","dispatched"),  "Shipped")
        .when(lower(trim(col("status"))).isin("pending","in progress"), "Pending")
        .when(lower(trim(col("status"))).isin("cancelled","canceled"),  "Cancelled")
        .when(lower(trim(col("status"))).isin("returned"),              "Returned")
        .otherwise(col("status"))
    )
)

(df_status_clean
    .write.format("delta")
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver_orders_s3"))

print("Status distribution after cleaning:")
spark.sql("SELECT status, COUNT(*) AS n FROM silver_orders_s3 GROUP BY status ORDER BY n DESC").show()
print("\n Saved as silver_orders_s3")


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 21, Finished, Available, Finished, False)

Status distribution after cleaning:
+---------+---+
|   status|  n|
+---------+---+
|Completed|354|
|  Shipped|287|
|  Pending|194|
|Cancelled|113|
| Returned| 52|
+---------+---+


 Saved as silver_orders_s3


---
## 🔗 Section 4 - Merging Queries (JOIN)

### What this does
We have three separate tables:
- **orders** - one row per order, with `customer_id` and `product_id` as foreign keys
- **customers** - customer profile data  
- **products** - product catalogue

We need to **merge** them into a single enriched dataset for reporting - the equivalent of
`Merge Queries` in Power Query, which performs a JOIN.

### M / Power Query equivalent
```
// Home -> Merge Queries
// Table 1: orders     -> Key column: customer_id
// Table 2: customers  -> Key column: customer_id
// Join kind: Left Outer
= Table.NestedJoin(orders, "customer_id", customers, "customer_id", "customers", JoinKind.LeftOuter)
// -> then expand the nested table, select columns
= Table.ExpandTableColumn(merged, "customers", {"full_name","country","segment"})
// -> repeat for products
```
> **Dataflows Gen2 note:** Each merge is a separate query step. You cannot easily control
> broadcast hints or partitioning strategies. With notebooks you have full control.


In [21]:
# ── Step 4.1  Load our three tables
df_o = spark.read.table("bronze_orders")
df_c = spark.read.table("bronze_customers")
df_p = spark.read.table("bronze_products")

print(f"Orders:    {df_o.count()} rows")
print(f"Customers: {df_c.count()} rows")
print(f"Products:  {df_p.count()} rows")


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 23, Finished, Available, Finished, False)

Orders:    1000 rows
Customers: 300 rows
Products:  94 rows


In [22]:
# ── Step 4.2  Simple LEFT JOIN — orders + customers
#
# LEFT JOIN = keep all orders, attach customer info where available
# (same as 'Left Outer' join kind in Power Query)

from pyspark.sql.functions import col

df_merged = (df_o
    .join(
        df_c.select("customer_id","full_name","country","region","segment"),
        on="customer_id",
        how="left"    # options: "left", "right", "inner", "outer", "cross", "semi", "anti"
    )
)

print(f"Merged rows: {df_merged.count()}")  # should equal orders count — it's a left join
df_merged.select("order_id","customer_id","full_name","country","segment","amount").show(5, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 24, Finished, Available, Finished, False)

Merged rows: 1000
+----------+-----------+------------------+-----------+-------+------+
|order_id  |customer_id|full_name         |country    |segment|amount|
+----------+-----------+------------------+-----------+-------+------+
|ORD-000018|CUST-0114  |Sofia Martin      |USA        |B2C    |56.39 |
|ORD-000157|CUST-0244  |Elizabeth Ferreira|Brazil     |B2B    |88.57 |
|ORD-000174|CUST-0063  |EVELYN tanaka     |Sweden     |B2C    |249.44|
|ORD-000290|CUST-0220  |Madison Jackson   |Canada     |B2B    |158.57|
|ORD-000551|CUST-0165  |Kenji Yamamoto    |South Korea|B2B    |149.11|
+----------+-----------+------------------+-----------+-------+------+
only showing top 5 rows



In [23]:
# ── Step 4.3  Chain a second JOIN - add product details
#
# Note: we rename the 'region' column in products to avoid column name collision
# (both orders and products had a 'region' concept; we only want the order region)

df_enriched = (df_o
    # Join 1: attach customer info
    .join(
        df_c.select("customer_id","full_name","country",
                    col("region").alias("customer_region"), "segment"),
        on="customer_id",
        how="left"
    )
    # Join 2: attach product catalogue info
    .join(
        df_p.select("product_id","category","subcategory",
                    col("price").alias("catalogue_price"), "supplier_info"),
        on="product_id",
        how="left"
    )
)

print(f"Enriched row count: {df_enriched.count()}")
print(f"Columns: {df_enriched.columns}")
df_enriched.select("order_id","full_name","segment","category","amount","catalogue_price").show(5)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 25, Finished, Available, Finished, False)

Enriched row count: 1000
Columns: ['product_id', 'customer_id', 'order_id', 'product_info', 'shipping_info', 'order_date', 'quantity', 'unit_price', 'discount_pct', 'amount', 'status', 'region', 'channel', 'payment_method', 'updated_at', 'full_name', 'country', 'customer_region', 'segment', 'category', 'subcategory', 'catalogue_price', 'supplier_info']
+----------+------------------+-------+-------------+------+---------------+
|  order_id|         full_name|segment|     category|amount|catalogue_price|
+----------+------------------+-------+-------------+------+---------------+
|ORD-000018|      Sofia Martin|    B2C|Home & Garden| 56.39|           54.5|
|ORD-000157|Elizabeth Ferreira|    B2B|Home & Garden| 88.57|           54.5|
|ORD-000174|     EVELYN tanaka|    B2C|Home & Garden|249.44|           54.5|
|ORD-000290|   Madison Jackson|    B2B|Home & Garden|158.57|           54.5|
|ORD-000551|    Kenji Yamamoto|    B2B|Home & Garden|149.11|           54.5|
+----------+-----------------

In [ ]:
# ── Step 4.4  Validate the join - check for unmatched keys (data quality check)
#
# In Power Query you'd notice this when the expanded table has nulls -> hard to spot
# In PySpark we can check explicitly

unmatched_customers = df_enriched.filter(col("full_name").isNull()).count()
unmatched_products  = df_enriched.filter(col("catalogue_price").isNull()).count()

print(f"Orders with no matching customer  : {unmatched_customers}")
print(f"Orders with no matching product   : {unmatched_products}")

if unmatched_customers > 0:
    print("\nSample unmatched customer IDs:")
    df_enriched.filter(col("full_name").isNull()).select("order_id","customer_id").show(5)


In [ ]:
# ── Step 4.5  Save the enriched 'gold' orders table

(df_enriched
    .write.format("delta")
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold_orders_enriched"))

print("Saved as gold_orders_enriched")
spark.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT category) AS categories, "
          "SUM(amount) AS total_revenue FROM gold_orders_enriched").show()


---
## ➕ Section 5 - Appending Queries (UNION / Stack)

### What this does
We want to combine `orders` (Jan–Nov 2024) with `orders_new` (Dec 2024 + Jan 2025) into
one unified table - stacking them vertically.  
This is the equivalent of `Append Queries` in Power Query.

### M / Power Query equivalent
```
// Home -> Append Queries -> Append as New
// Table 1: orders
// Table 2: orders_new
// -> creates a new query that UNION-stacks both tables
= Table.Combine({orders, orders_new})
```
> **Dataflows Gen2 note:** Append works well for simple stacking.  
> The limitation appears when you need deduplication, ordering, or schema reconciliation - 
> that's where PySpark gives you full control.


In [24]:
# ── Step 5.1  Inspect both DataFrames before appending
df_hist = spark.read.table("bronze_orders")
df_new  = spark.read.csv("Files/orders_new.csv", header=True, inferSchema=True)

print(f"Historical orders : {df_hist.count()} rows")
print(f"New orders        : {df_new.count()} rows")

print("\nDate range — historical:")
df_hist.selectExpr("MIN(order_date) AS earliest","MAX(order_date) AS latest").show()

print("Date range — new:")
df_new.selectExpr("MIN(order_date) AS earliest","MAX(order_date) AS latest").show()


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 26, Finished, Available, Finished, False)

Historical orders : 1000 rows
New orders        : 200 rows

Date range — historical:
+----------+----------+
|  earliest|    latest|
+----------+----------+
|2024-01-02|2024-11-30|
+----------+----------+

Date range — new:
+----------+----------+
|  earliest|    latest|
+----------+----------+
|2024-01-07|2025-01-31|
+----------+----------+



In [25]:
# ── Step 5.2  Simple UNION (append all rows)
#
# unionByName()  -> matches columns by NAME, not position (safer than union())
#                  handles column order differences between the two DataFrames
# allowMissingColumns=True -> fills missing columns with null instead of failing

df_all = df_hist.unionByName(df_new, allowMissingColumns=True)

print(f"Combined row count: {df_all.count()}")
print(f"(historical {df_hist.count()} + new {df_new.count()} = {df_hist.count()+df_new.count()})")
print("\nFull date range after append:")
df_all.selectExpr("MIN(order_date) AS earliest","MAX(order_date) AS latest").show()


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 27, Finished, Available, Finished, False)

Combined row count: 1200
(historical 1000 + new 200 = 1200)

Full date range after append:
+----------+----------+
|  earliest|    latest|
+----------+----------+
|2024-01-02|2025-01-31|
+----------+----------+



In [26]:
# ── Step 5.3  Append WITH deduplication
#
# orders_new contains ~50 updated records (same order_id, newer updated_at)
# We want to keep the LATEST version of each order_id

from pyspark.sql.functions import row_number, desc
from pyspark.sql.window import Window

# Window: partition by order_id, rank by updated_at descending → rank 1 = latest
w = Window.partitionBy("order_id").orderBy(desc("updated_at"))

df_deduped = (df_all
    .withColumn("_rn", row_number().over(w))  # rank each order version
    .filter(col("_rn") == 1)                  # keep only the latest
    .drop("_rn")
)

print(f"After deduplication: {df_deduped.count()} rows")
print(f"Distinct order IDs : {df_deduped.select('order_id').distinct().count()}")

# Spot check: did the status update come through?
updated_sample = df_new.select("order_id","status","updated_at").limit(5)
print("\nSample updated records (from orders_new):")
updated_sample.show(truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 28, Finished, Available, Finished, False)

After deduplication: 1150 rows
Distinct order IDs : 1150

Sample updated records (from orders_new):
+----------+---------+-------------------+
|order_id  |status   |updated_at         |
+----------+---------+-------------------+
|ORD-001118|Shipped  |2025-01-13 06:00:00|
|ORD-000186|Completed|2025-01-08 20:00:00|
|ORD-001064|Shipped  |2024-12-31 18:00:00|
|ORD-000406|Cancelled|2024-12-19 12:00:00|
|ORD-000275|Cancelled|2025-01-09 08:00:00|
+----------+---------+-------------------+



In [27]:
# ── Step 5.4  Save the unified orders table

(df_deduped
    .write.format("delta")
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver_orders_unified"))

print("Saved as silver_orders_unified")
spark.sql("""
    SELECT 
        DATE_FORMAT(order_date,'yyyy-MM') AS month,
        COUNT(*) AS orders,
        ROUND(SUM(amount),2) AS revenue
    FROM silver_orders_unified
    GROUP BY 1 ORDER BY 1
""").show(20)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 29, Finished, Available, Finished, False)

Saved as silver_orders_unified
+-------+------+---------+
|  month|orders|  revenue|
+-------+------+---------+
|2024-01|    85| 79116.55|
|2024-02|    79| 99174.39|
|2024-03|   103|105804.77|
|2024-04|   101|156983.18|
|2024-05|    82| 94474.57|
|2024-06|    74| 87373.01|
|2024-07|   112|  98273.3|
|2024-08|    85|142188.47|
|2024-09|    91| 88964.17|
|2024-10|    96| 95307.06|
|2024-11|    92| 87178.16|
|2024-12|    78| 72381.53|
|2025-01|    72| 59778.19|
+-------+------+---------+



---
## 🔀 Section 6 - Pivoting

### What this does
We have order data in **long format** - one row per order.  
We want to produce a **pivot table**: one row per category, one column per region,  
values = total revenue.

```
Long format (before):          Pivot (after):
category   | region   | amt    category  | Europe | N.America | Asia Pac...
Electronics| Europe   | 999    Electronics| 45000 | 87000     | 23000  ...
Clothing   | N.America| 149    Clothing   | 12000 | 33000     | 9000   ...
Electronics| N.America| 249    ...
```

### M / Power Query equivalent
```
// Transform -> Pivot Column
// Values column  : amount
// Aggregate      : Sum
// -> produces one column per unique region value (dynamic - adds new columns if new regions appear)
= Table.Pivot(
    grouped_table,
    List.Distinct(grouped_table[region]),
    "region",
    "total_amount",
    List.Sum
)
```


In [28]:
# ── Step 6.1  Prepare the aggregated source (long format)
from pyspark.sql.functions import round as spark_round, sum as spark_sum

df_long = (spark.read.table("silver_orders_unified")
    # First split product_info so we have category
    .withColumn("category", trim(split(col("product_info"), r"\|").getItem(0)))
    .groupBy("category", "region")
    .agg(spark_round(spark_sum("amount"), 2).alias("total_revenue"))
)

print("Long format (source for pivot):")
df_long.orderBy("category","region").show(20)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 30, Finished, Available, Finished, False)

Long format (source for pivot):
+-------------+--------------------+-------------+
|     category|              region|total_revenue|
+-------------+--------------------+-------------+
|Books & Media|        Asia Pacific|      3264.01|
|Books & Media|              Europe|      4895.72|
|Books & Media|       Latin America|      1876.18|
|Books & Media|Middle East & Africa|       635.63|
|Books & Media|       North America|      5400.62|
|     Clothing|        Asia Pacific|      11810.0|
|     Clothing|              Europe|     12394.08|
|     Clothing|       Latin America|      6040.43|
|     Clothing|Middle East & Africa|      2849.72|
|     Clothing|       North America|     20455.38|
|  Electronics|        Asia Pacific|     156435.9|
|  Electronics|              Europe|    321370.22|
|  Electronics|       Latin America|     63239.99|
|  Electronics|Middle East & Africa|     56422.21|
|  Electronics|       North America|    389502.88|
|Home & Garden|        Asia Pacific|     30795.57|

In [29]:
# ── Step 6.2  Pivot: rows = category, columns = region, values = total_revenue
#
# .groupBy(row_dimension)
# .pivot(column_dimension)   <- this is the key method
# .agg(aggregation)

df_pivot = (df_long
    .groupBy("category")           # rows
    .pivot("region")               # columns — one per unique region value
    .agg(spark_round(spark_sum("total_revenue"), 2).alias("revenue"))
    .fillna(0)                     # replace null (no sales in that cell) with 0
    .orderBy("category")
)

print("Pivot result:")
df_pivot.show(truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 31, Finished, Available, Finished, False)

Pivot result:
+-----------------+------------+---------+-------------+--------------------+-------------+
|category         |Asia Pacific|Europe   |Latin America|Middle East & Africa|North America|
+-----------------+------------+---------+-------------+--------------------+-------------+
|Books & Media    |3264.01     |4895.72  |1876.18      |635.63              |5400.62      |
|Clothing         |11810.0     |12394.08 |6040.43      |2849.72             |20455.38     |
|Electronics      |156435.9    |321370.22|63239.99     |56422.21            |389502.88    |
|Home & Garden    |30795.57    |36378.52 |15105.39     |8684.16             |53583.68     |
|Sports & Outdoors|10934.7     |21832.63 |6812.45      |2569.12             |23708.16     |
+-----------------+------------+---------+-------------+--------------------+-------------+



In [ ]:
# ── Step 6.3  Performance tip: specify pivot values explicitly
#
# Without a value list, Spark must do TWO passes over the data (one to find distinct values)
# Providing the list means ONE pass - ~2x faster on large datasets

known_regions = ["Europe", "North_America", "Asia_Pacific", "Latin_America", "Middle_East_&_Africa"]

df_pivot_fast = (df_long
    .groupBy("category")
    .pivot("region", known_regions)    # <- provide values list here
    .agg(spark_round(spark_sum("total_revenue"), 2))
    .fillna(0)
    .orderBy("category")
)

df_pivot_fast.show(truncate=False)
print("\n Equivalent result — but one Spark job instead of two")


In [ ]:
# ── Step 6.4  Save pivot as a reporting table

(df_pivot_fast
    .write.format("delta")
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold_revenue_by_category_region"))

print(" Saved as gold_revenue_by_category_region")


---
## 🔃 Section 7 - Unpivoting (Wide -> Long / Melt)

### What this does
We have a **wide** table from the BI team: one row per region, one column per month.  
We need to transform it into **long** format (one row per region+month) so it can be
loaded into a fact table or used in a time-series model.

```
Wide (before):                          Long (after):
region      | Jan_2024 | Feb_2024 ...   region      | month    | revenue
Europe      | 125000   | 134000  ...    Europe      | Jan_2024 | 125000
N.America   | 210000   | 198000  ...    Europe      | Feb_2024 | 134000
                                        N.America   | Jan_2024 | 210000 ...
```

### M / Power Query equivalent
```
// Select the month columns -> Transform -> Unpivot Columns
= Table.Unpivot(
    Source,
    {"Jan_2024","Feb_2024",...,"Dec_2024"},   // columns to unpivot
    "month",                                   // attribute column name
    "revenue"                                  // value column name
)
```


In [30]:
# ── Step 7.1  Load the wide table and inspect
df_wide = spark.read.csv("Files/sales_wide.csv", header=True, inferSchema=True)
print("Wide format (source):")
df_wide.show(truncate=False)
print(f"Shape: {df_wide.count()} rows × {len(df_wide.columns)} columns")


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 32, Finished, Available, Finished, False)

Wide format (source):
+-------------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|region       |Jan_2024 |Feb_2024 |Mar_2024 |Apr_2024 |May_2024 |Jun_2024 |Jul_2024 |Aug_2024 |Sep_2024 |Oct_2024 |Nov_2024 |Dec_2024 |
+-------------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|North_America|208049.72|259448.15|214520.39|208865.1 |202044.24|262330.43|155038.68|255080.39|176423.73|181574.52|186571.68|226655.55|
|Europe       |113945.2 |117316.54|113397.86|94920.75 |116124.01|74360.35 |99167.14 |69936.8  |73614.36 |115456.62|102980.95|99946.66 |
|Asia_Pacific |116421.31|107630.06|160173.78|115463.62|110367.52|117012.72|151857.18|99743.63 |131177.41|110354.87|119844.64|152564.01|
|Latin_America|250540.32|359243.51|315710.68|294237.6 |348320.39|218392.36|278699.79|267351.54|309382.39|345635.79|282088.83|348356.7 |
|MEA          |205818.07|1

In [31]:
# ── Step 7.2  Unpivot using stack() — the PySpark equivalent of melt/unpivot
#
# stack(n, 'col1', val1, 'col2', val2, ...) is a SQL generator function that
# creates n new rows per original row
# It's the most efficient approach in PySpark (single pass)

from pyspark.sql.functions import expr

# Build the month columns list programmatically (no hardcoding)
month_cols = [c for c in df_wide.columns if c != "region"]
print(f"Month columns to unpivot: {month_cols}")

# Build the stack() expression
stack_expr = f"stack({len(month_cols)}, " + ", ".join([f"'{c}', `{c}`" for c in month_cols]) + ") AS (month, revenue)"

df_long = (df_wide
    .select("region", expr(stack_expr))
    .filter(col("revenue").isNotNull())
    .orderBy("region","month")
)

print(f"\nLong format: {df_long.count()} rows (expected: {len(month_cols) * df_wide.count()})")
df_long.show(15, truncate=False)


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 33, Finished, Available, Finished, False)

Month columns to unpivot: ['Jan_2024', 'Feb_2024', 'Mar_2024', 'Apr_2024', 'May_2024', 'Jun_2024', 'Jul_2024', 'Aug_2024', 'Sep_2024', 'Oct_2024', 'Nov_2024', 'Dec_2024']

Long format: 60 rows (expected: 60)
+------------+--------+---------+
|region      |month   |revenue  |
+------------+--------+---------+
|Asia_Pacific|Apr_2024|115463.62|
|Asia_Pacific|Aug_2024|99743.63 |
|Asia_Pacific|Dec_2024|152564.01|
|Asia_Pacific|Feb_2024|107630.06|
|Asia_Pacific|Jan_2024|116421.31|
|Asia_Pacific|Jul_2024|151857.18|
|Asia_Pacific|Jun_2024|117012.72|
|Asia_Pacific|Mar_2024|160173.78|
|Asia_Pacific|May_2024|110367.52|
|Asia_Pacific|Nov_2024|119844.64|
|Asia_Pacific|Oct_2024|110354.87|
|Asia_Pacific|Sep_2024|131177.41|
|Europe      |Apr_2024|94920.75 |
|Europe      |Aug_2024|69936.8  |
|Europe      |Dec_2024|99946.66 |
+------------+--------+---------+
only showing top 15 rows



In [32]:
# ── Step 7.3  Clean up month column — convert 'Jan_2024' -> date '2024-01-01'
#
# This makes the data usable in time-series analysis and date filtering

from pyspark.sql.functions import to_date, regexp_replace, concat, lit

# Map short month names to numbers
month_map_expr = (
    "CASE "
    "WHEN month LIKE 'Jan%' THEN '01' WHEN month LIKE 'Feb%' THEN '02' "
    "WHEN month LIKE 'Mar%' THEN '03' WHEN month LIKE 'Apr%' THEN '04' "
    "WHEN month LIKE 'May%' THEN '05' WHEN month LIKE 'Jun%' THEN '06' "
    "WHEN month LIKE 'Jul%' THEN '07' WHEN month LIKE 'Aug%' THEN '08' "
    "WHEN month LIKE 'Sep%' THEN '09' WHEN month LIKE 'Oct%' THEN '10' "
    "WHEN month LIKE 'Nov%' THEN '11' WHEN month LIKE 'Dec%' THEN '12' "
    "END"
)

from pyspark.sql.functions import regexp_extract

df_long_clean = (df_long
    .withColumn("year",       regexp_extract(col("month"), r"(\d{4})", 1))
    .withColumn("month_num",  expr(month_map_expr))
    .withColumn("month_date", to_date(concat(col("year"), lit("-"), col("month_num"), lit("-01")), "yyyy-MM-dd"))
    .drop("year","month_num")
    .orderBy("region","month_date")
)

df_long_clean.show(15)

# Save
(df_long_clean
    .write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("gold_monthly_sales"))
print("\n✅ Saved as gold_monthly_sales")


StatementMeta(, 92b1786b-1972-4abb-956e-127db8f815ad, 34, Finished, Available, Finished, False)

+------------+--------+---------+----------+
|      region|   month|  revenue|month_date|
+------------+--------+---------+----------+
|Asia_Pacific|Jan_2024|116421.31|2024-01-01|
|Asia_Pacific|Feb_2024|107630.06|2024-02-01|
|Asia_Pacific|Mar_2024|160173.78|2024-03-01|
|Asia_Pacific|Apr_2024|115463.62|2024-04-01|
|Asia_Pacific|May_2024|110367.52|2024-05-01|
|Asia_Pacific|Jun_2024|117012.72|2024-06-01|
|Asia_Pacific|Jul_2024|151857.18|2024-07-01|
|Asia_Pacific|Aug_2024| 99743.63|2024-08-01|
|Asia_Pacific|Sep_2024|131177.41|2024-09-01|
|Asia_Pacific|Oct_2024|110354.87|2024-10-01|
|Asia_Pacific|Nov_2024|119844.64|2024-11-01|
|Asia_Pacific|Dec_2024|152564.01|2024-12-01|
|      Europe|Jan_2024| 113945.2|2024-01-01|
|      Europe|Feb_2024|117316.54|2024-02-01|
|      Europe|Mar_2024|113397.86|2024-03-01|
+------------+--------+---------+----------+
only showing top 15 rows


✅ Saved as gold_monthly_sales


---
## 🔡 Section 8 - Uppercase / Lowercase Transformations

### What this does
The `customers` table has inconsistent casing in `full_name` and `email`:
- Names like `"  BENJAMIN Smith"`, `"lucas mia"`, `"  Ethan  "` 
- Emails with mixed case prefixes

We'll harmonize:
- Names -> **Title Case** (first letter of each word capitalised)  
- Emails -> **all lowercase** (emails are case-insensitive but conventionally lowercase)
- Category codes -> **UPPERCASE** (for consistent key matching)

### M / Power Query equivalent
```
// Transform tab → Format
// UPPERCASE  : Table.TransformColumns(Source, {{"col", Text.Upper}})
// lowercase  : Table.TransformColumns(Source, {{"col", Text.Lower}})
// Title Case : Table.TransformColumns(Source, {{"col", Text.Proper}})
```


In [ ]:
# ── Step 8.1  Audit the messy name/email data
print("Sample raw customer names and emails:")
spark.read.table("bronze_customers") \
    .select("customer_id","full_name","email") \
    .show(12, truncate=False)


In [ ]:
# ── Step 8.2  Lowercase - emails
from pyspark.sql.functions import lower, upper, initcap, trim, regexp_replace

df_c = spark.read.table("bronze_customers")

df_lower = df_c.withColumn("email", lower(trim(col("email"))))

print("After lower(trim(email)):")
df_lower.select("customer_id","email").show(8, truncate=False)


In [ ]:
# ── Step 8.3  Title Case - names
#
# initcap() -> capitalises the first letter of each word (Power Query's Text.Proper equivalent)
# We also trim extra whitespace before applying initcap

df_title = (df_c
    .withColumn("full_name",
        initcap(                           # Title Case each word
            regexp_replace(                # collapse multiple spaces to one
                trim(col("full_name")),    # trim leading/trailing whitespace first
                r"\s+", " "
            )
        )
    )
    .withColumn("email", lower(trim(col("email"))))
)

print("After initcap(trim(full_name)):")
df_title.select("customer_id","full_name","email").show(12, truncate=False)


In [ ]:
# ── Step 8.4  Uppercase - useful for key columns / category codes
#
# upper() is handy when joining on category codes or country codes
# where source systems may use different casing

df_keys = (spark.read.table("bronze_orders")
    .withColumn("region_key",  upper(trim(col("region"))))
    .withColumn("channel_key", upper(trim(col("channel"))))
)

print("Uppercase key columns:")
df_keys.select("order_id","region","region_key","channel","channel_key").show(6)


In [ ]:
# ── Step 8.5  Save cleaned customers

(df_title
    .write.format("delta")
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver_customers_clean"))

print(" Saved as silver_customers_clean")
spark.sql("SELECT customer_id, full_name, email FROM silver_customers_clean LIMIT 5").show(truncate=False)


---
## ✂️ Section 9 - Trimming Whitespace

### What this does
Leading and trailing whitespace is the most common cause of failed JOINs and
incorrect GROUP BY results. The `customers.full_name` and `customers.email` columns  
have been intentionally dirtied with leading/trailing/internal spaces.

We'll demonstrate:
- `trim()` - remove leading AND trailing whitespace  
- `ltrim()` / `rtrim()` - left-only or right-only
- `regexp_replace(..., r"\s+", " ")` - collapse internal multiple spaces to one
- Combined: trim + collapse (the real-world recipe)

### M / Power Query equivalent
```
// Transform -> Format -> Trim
// = Table.TransformColumns(Source, {{"full_name", Text.Trim}})
// Note: Power Query's Text.Trim only removes leading/trailing - 
// it does NOT collapse internal spaces. PySpark lets you do both.
```


In [ ]:
# ── Step 9.1  Show the problem — hidden spaces
from pyspark.sql.functions import length, trim, ltrim, rtrim, regexp_replace

df_c = spark.read.table("bronze_customers")

# Use length comparison to expose hidden whitespace
df_c.select(
    col("customer_id"),
    col("full_name"),
    length(col("full_name")).alias("raw_len"),
    length(trim(col("full_name"))).alias("trimmed_len")
).filter(
    length(col("full_name")) != length(trim(col("full_name")))
).show(10, truncate=False)


In [ ]:
# ── Step 9.2  Demonstrate each trim variant

sample = df_c.select("customer_id","full_name","email").limit(15)

print("trim()  — removes leading AND trailing whitespace:")
sample.withColumn("name_trim", trim(col("full_name"))).show(5, truncate=False)

print("ltrim() — removes LEADING whitespace only:")
sample.withColumn("name_ltrim", ltrim(col("full_name"))).show(5, truncate=False)

print("rtrim() — removes TRAILING whitespace only:")
sample.withColumn("name_rtrim", rtrim(col("full_name"))).show(5, truncate=False)


In [ ]:
# ── Step 9.3  Full clean recipe: trim + collapse internal spaces + title case
#
# This is the production-grade pattern:
# 1. trim()              -> remove leading/trailing spaces
# 2. regexp_replace \s+  -> collapse multiple internal spaces to one
# 3. initcap()           -> normalise casing while we're here

def clean_text_column(df, col_name):
    """Remove leading/trailing/internal extra whitespace and apply title case."""
    return df.withColumn(col_name,
        initcap(
            regexp_replace(
                trim(col(col_name)),
                r"\s+", " "
            )
        )
    )

df_trim = (df_c
    .transform(lambda df: clean_text_column(df, "full_name"))
    .withColumn("email", lower(trim(col("email"))))
)

print("Before vs After:")
df_c.select("full_name","email").show(6, truncate=False)
print("↓ After clean_text_column ↓")
df_trim.select("full_name","email").show(6, truncate=False)


In [ ]:
# ── Step 9.4  Validate — no more hidden whitespace

issues_before = df_c.filter(length(col("full_name")) != length(trim(col("full_name")))).count()
issues_after  = df_trim.filter(length(col("full_name")) != length(trim(col("full_name")))).count()

print(f"Customers with whitespace issues  BEFORE: {issues_before}")
print(f"Customers with whitespace issues  AFTER : {issues_after}")
print()

(df_trim
    .write.format("delta")
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver_customers_trimmed"))
print("Saved as silver_customers_trimmed")


---
## ⏱️ Section 10 - Incremental Loading

### What this does
Loading 1 000+ rows on every run is inefficient. We want to **only process new or changed records**.  

We demonstrate two complementary patterns:

| Pattern | Use case | Dataflows Gen2 equivalent |
|---------|----------|--------------------------|
| **Watermark + Append** | Append-only streams (new orders only) | Incremental Refresh (bucket replace) |
| **Delta MERGE** | Upsert - new rows AND row-level updates | ❌ Not supported in Dataflows Gen2 |

### DataFlows Gen2 Incremental Refresh - how it works (and its limits)
```
// In DataFlows Gen2:
// Settings -> Incremental Refresh
//   • DateTime column  : updated_at
//   • Bucket size      : 1 day
//   • Window           : last 30 days
//   • Update method    : Replace bucket (delete + re-insert ALL rows in changed bucket)
//
// ⚠ Requires query folding (source must support push-down)
// ⚠ Update method = bucket replace -> NOT row-level upsert
// ⚠ Max 50 buckets per query, 150 per dataflow
// ⚠ OPTIMIZE / VACUUM blocked on Incremental Refresh tables
```

> **Notebook advantage:** true row-level MERGE, no folding requirement,
> full OPTIMIZE + V-Order control after every load.


In [ ]:
%%sql

CREATE TABLE silver_orders_incremental
AS
SELECT *
FROM bronze_orders

In [ ]:
# ── Step 10.1  Watermark pattern — append only NEW rows
#
# The watermark is the MAX(updated_at) we have already loaded.
# On every run: load only rows where updated_at > watermark.

from pyspark.sql.functions import col, max as spark_max
from datetime import datetime

# Simulate: we already have data up to 2024-11-30
# In production this watermark comes from reading the last loaded table

watermark = spark.sql(
    "SELECT COALESCE(MAX(updated_at), '1900-01-01 00:00:00') AS wm FROM silver_orders_incremental"
).collect()[0]["wm"]

print(f"Current watermark (MAX updated_at in existing table): {watermark}")

# Load only rows newer than the watermark
df_incremental = (spark.read
    .csv("Files/orders_new.csv", header=True, inferSchema=True)
    .filter(col("updated_at") > watermark)
)

print(f"Rows to load this run: {df_incremental.count()}")
df_incremental.select("order_id","order_date","updated_at","status").show(5)


In [ ]:
# ── Step 10.2  Append new rows to the silver_orders_incremental table
(df_incremental
    .write
    .format("delta")
    .mode("append")          # append, not overwrite
    .saveAsTable("silver_orders_incremental"))

new_watermark = spark.sql("SELECT MAX(updated_at) FROM silver_orders_incremental").collect()[0][0]
total         = spark.sql("SELECT COUNT(*) FROM silver_orders_incremental").collect()[0][0]

print(f"New watermark after load: {new_watermark}")
print(f"Total rows in table     : {total}")


In [ ]:
# ── Step 10.3  Delta MERGE - true upsert (INSERT new + UPDATE changed)
#
# This is the pattern Dataflows Gen2 cannot do.
# We process ALL rows from orders_new: 
#   • If order_id already exists -> UPDATE the record
#   • If order_id is new         -> INSERT it

from delta.tables import DeltaTable

# Ensure the target table exists (use silver_orders_incremental created above)
target = DeltaTable.forName(spark, "silver_orders_incremental")

source = spark.read.csv("Files/orders_new.csv", header=True, inferSchema=True)

print(f"Source rows: {source.count()}")
print(f"Target rows BEFORE merge: {spark.sql('SELECT COUNT(*) FROM silver_orders_incremental').collect()[0][0]}")

(target.alias("t")
    .merge(
        source.alias("s"),
        "t.order_id = s.order_id"    # match condition
    )
    .whenMatchedUpdateAll()          # UPDATE: all columns from source
    .whenNotMatchedInsertAll()       # INSERT: all columns for new rows
    .execute()
)

print(f"Target rows AFTER merge : {spark.sql('SELECT COUNT(*) FROM silver_orders_incremental').collect()[0][0]}")
print("\n MERGE complete - rows updated and inserted without full reload")


In [ ]:
# ── Step 10.4  Verify the MERGE - spot-check a few updated records

print("Sample of records that were updated (status changes from orders_new):")
spark.sql("""
    SELECT t.order_id, t.status, t.updated_at
    FROM silver_orders_incremental t
    INNER JOIN (
        SELECT order_id, MAX(updated_at) AS latest_update
        FROM silver_orders_incremental
        WHERE order_date < '2024-12-01'   -- original orders that got updated
        GROUP BY order_id
    ) upd ON t.order_id = upd.order_id
    WHERE t.updated_at > '2024-11-30'
    ORDER BY t.updated_at DESC
    LIMIT 10
""").show(truncate=False)


In [ ]:
# ── Step 10.5  Delta time travel - see the table history

print("Delta table history (audit trail — free with Delta, zero extra work):")
spark.sql("DESCRIBE HISTORY silver_orders_incremental").select(
    "version","timestamp","operation","operationMetrics"
).show(5, truncate=False)

# You can read any previous version:
# df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("silver_orders_incremental")
print("\n Tip: use spark.read.format('delta').option('versionAsOf', 0).table('...') to read any past version")


---
## ⚡ Section 11 - Engine Optimisations

### What this does
After all our transformations, we need to make sure the Delta tables are *fast to read*.  
Five complementary techniques - use them together as part of your standard load pattern:

| Technique | When to use | Benefit |
|-----------|-------------|---------|
| **V-Order** | Every write to a Gold/Silver table | 40–60% faster Power BI Direct Lake reads |
| **OPTIMIZE** (bin-compact) | After append-heavy loads | Merges small files → faster scans |
| **VACUUM** | Scheduled (weekly) | Removes old Delta versions → saves storage |
| **DataFrame caching** | Reference/dimension tables read many times | Eliminates repeated Spark reads |
| **Broadcast join** | Small table joined to a large one | Eliminates shuffle → much faster joins |

> **Dataflows Gen2:** None of these are available, except V-Order on the written Delta tables in OneLake. OPTIMIZE is blocked on Incremental Refresh
> tables. Caching has no equivalent. This is the biggest performance gap.


### 11a - V-Order

**What it is:** A write-time optimisation for Parquet files, built into Fabric's Spark runtime.  
Applies VertiPaq-compatible sorting, encoding and compression so that Power BI, SQL endpoint  
and Spark all read your Delta tables faster.

**Performance impact:**
- Power BI Direct Lake cold-cache: **40–60% faster**
- SQL Analytics Endpoint: **~10% faster** 
- Spark reads: marginal direct gain, but combined with OPTIMIZE the effect compounds

**100% open-source Delta Lake compliant** - files are readable by any Delta Lake reader.


In [ ]:
# ── Step 11a.1  Check current V-Order setting
current = spark.conf.get("spark.sql.parquet.vorder.enabled", "not set")
print(f"V-Order currently enabled: {current}")

In [ ]:
# ── Step 11a.2  Enable V-Order for the entire session (if not already set)
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
print("V-Order enabled for this session ✓")

# ── Step 11a.3  Set V-Order permanently on a specific table
spark.sql("""
    ALTER TABLE gold_orders_enriched
    SET TBLPROPERTIES ('delta.parquet.vorder.enabled' = 'true')
""")
print("V-Order set as permanent table property on gold_orders_enriched ✓")


In [ ]:
# ── Step 11a.4  Apply V-Order retroactively with OPTIMIZE
# Any existing Parquet files that were NOT written with V-Order can be rewritten:

print("Running OPTIMIZE with V-Order on gold_orders_enriched...")
spark.sql("OPTIMIZE gold_orders_enriched VORDER")
print("✓ Done — all Parquet files now V-Order optimised")

# You can combine with Z-Order if you also want data colocation by a filter column:
# spark.sql("OPTIMIZE gold_orders_enriched ZORDER BY (category, region) VORDER")
print("\n Tip: ZORDER and VORDER are compatible — use both if you have common filter columns")


### 11b - OPTIMIZE (Bin-Compaction)

**What it is:** Merges many small Parquet files into fewer, larger files (target ~128 MB – 1 GB each).  
Append-heavy workloads (like incremental loads) create many small files over time.  
Reading 500 files of 1 MB each is much slower than reading 2 files of 250 MB each.

**When to run:** After every significant append or MERGE operation.  
For very high-frequency loads, run it on a schedule (e.g. every 4–6 hours).


In [ ]:
# ── Step 11b.1  Check file count BEFORE OPTIMIZE
from delta.tables import DeltaTable

detail = spark.sql("DESCRIBE DETAIL silver_orders_unified").collect()[0]
print(f"Table size    : {detail['sizeInBytes'] / 1024:.1f} KB")
print(f"File count    : {detail['numFiles']}")


In [ ]:
# ── Step 11b.2  Run OPTIMIZE (+ V-Order in one command)
print("Running OPTIMIZE on silver_orders_unified...")
spark.sql("OPTIMIZE silver_orders_unified VORDER")
print("✓ Done")

# Check after
detail_after = spark.sql("DESCRIBE DETAIL silver_orders_unified").collect()[0]
print(f"\nAfter OPTIMIZE:")
print(f"Table size    : {detail_after['sizeInBytes'] / 1024:.1f} KB")
print(f"File count    : {detail_after['numFiles']}")


In [ ]:
# ── Step 11b.3  OPTIMIZE with Z-Order - collocate data by common filter columns
#
# Z-Order physically sorts data within each file by the specified columns.
# Queries that filter by those columns skip most files entirely (data skipping).
# Best for: columns frequently used in WHERE clauses or JOIN conditions.

print("Running OPTIMIZE + ZORDER BY (category, region) on gold_orders_enriched...")
spark.sql("OPTIMIZE gold_orders_enriched ZORDER BY (category, region) VORDER")
print("✓ Done - data colocated by category + region, V-Order applied")


### 11c - VACUUM

**What it is:** Removes Parquet files that are no longer referenced by any Delta version  
(i.e. files left over from old MERGE / OPTIMIZE operations).

**Why it matters:** Every MERGE, OPTIMIZE, or overwrite write creates new files.
Old files are kept so you can do time-travel reads (`versionAsOf`, `timestampAsOf`).
Without VACUUM, storage grows indefinitely.

**Default retention: 7 days (168 hours)** - don't go below this in production unless
you're certain no long-running jobs are reading old versions.


In [ ]:
# ── Step 11c.1  Show current Delta history (versions available for time travel)
print("Current Delta history for silver_orders_unified:")
spark.sql("DESCRIBE HISTORY silver_orders_unified").select(
    "version","timestamp","operation"
).show(10, truncate=False)


In [ ]:
# ── Step 11c.2  VACUUM - remove files older than retention period
#
# DRY RUN first (recommended): shows what WOULD be deleted without deleting

print("VACUUM dry run (files that WOULD be removed):")
spark.sql("VACUUM silver_orders_unified RETAIN 168 HOURS DRY RUN").show(10, truncate=False)

# Actual VACUUM (uncomment when ready):
# spark.sql("VACUUM silver_orders_unified RETAIN 168 HOURS")
# print("✓ VACUUM complete")


In [ ]:
# ── Step 11c.3  Schedule VACUUM as part of your maintenance notebook
#
# Best practice: separate VACUUM into a weekly maintenance notebook
# rather than running it after every load (it's expensive and not needed every time)

tables_to_vacuum = [
    "silver_orders_unified",
    "gold_orders_enriched",
    "gold_revenue_by_category_region",
    "gold_monthly_sales",
]

print("VACUUM schedule (dry run):")
for table in tables_to_vacuum:
    try:
        result = spark.sql(f"VACUUM {table} RETAIN 168 HOURS DRY RUN")
        count = result.count()
        print(f"  {table}: {count} files eligible for removal")
    except Exception as e:
        print(f"  {table}: skipped ({str(e)[:60]})")


### 11d - DataFrame Caching

**What it is:** Pins a DataFrame in Spark's distributed memory (or disk) so repeated reads
don't re-scan the source files.

**When to use:**
- Dimension / reference tables read many times in a single notebook run  
- DataFrames used as the right side of multiple joins  
- Any DataFrame you call `.show()`, `.count()`, or `.filter()` on repeatedly

**When NOT to use:**  
- Very large fact tables (they won't fit in memory)  
- DataFrames only used once  
- Streaming scenarios


In [ ]:
# ── Step 11d.1  Cache a dimension table — products catalogue
#
# .cache() is lazy — it doesn't materialise until an action is called
# .count() triggers the materialisation (forces the cache to be populated)

from pyspark.sql.functions import broadcast

df_products_dim = spark.read.table("bronze_products").cache()
df_products_dim.count()    # trigger cache materialisation

print("df_products_dim is now cached in Spark memory ✓")
print(f"Product count: {df_products_dim.count()}")   # this read comes from cache — no file I/O


In [ ]:
# ── Step 11d.2  Demonstrate cache benefit with a timing comparison
import time

df_orders_large = spark.read.table("silver_orders_unified")

# Without cache — reads from Delta files every time
t0 = time.time()
df_products_nocache = spark.read.table("bronze_products")
for _ in range(3):
    df_orders_large.join(df_products_nocache, "product_id").count()
t_nocache = time.time() - t0

# With cache — reads from memory after first access
t0 = time.time()
for _ in range(3):
    df_orders_large.join(df_products_dim, "product_id").count()  # df_products_dim is cached
t_cache = time.time() - t0

print(f"3 joins WITHOUT cache: {t_nocache:.2f}s")
print(f"3 joins WITH    cache: {t_cache:.2f}s")
print(f"Speedup: {t_nocache/t_cache:.1f}x")


In [ ]:
# ── Step 11d.3  Unpersist when done - free up memory for other operations

df_products_dim.unpersist()
print("Cache released ✓")

# StorageLevel options (more granular control):
# from pyspark import StorageLevel
# df.persist(StorageLevel.MEMORY_AND_DISK)  # spill to disk if memory is full
# df.persist(StorageLevel.DISK_ONLY)        # disk only — for very large DataFrames


### 11e - Broadcast Joins

**What it is:** When joining a **large** table to a **small** table, Spark normally shuffles
both tables across the network (expensive). With a broadcast join, the small table is sent
to *every executor* in full, so only the large table moves - and only once.

**When to use:** Small table < a few hundred MB (dimension tables, lookup tables, reference data).  
**Rule of thumb:** if the table fits comfortably in the driver memory, it fits in a broadcast.

**Auto-broadcast threshold:** Spark auto-broadcasts tables smaller than `spark.sql.autoBroadcastJoinThreshold`  
(default 10 MB). For tables 10 MB–several hundred MB, hint manually.


In [ ]:
# ── Step 11e.1  Check table sizes
print("Table sizes:")
for t in ["bronze_products","bronze_customers","silver_orders_unified"]:
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {t}").collect()[0]
        kb = detail["sizeInBytes"] / 1024
        print(f"  {t:<35}: {kb:>8.1f} KB  ({detail['numFiles']} files)")
    except:
        print(f"  {t}: could not read detail")


In [ ]:
# ── Step 11e.2  Join WITHOUT broadcast hint (Spark decides - may do a shuffle)
from pyspark.sql.functions import broadcast, col, count as spark_count
import time

df_orders = spark.read.table("silver_orders_unified")
df_prods  = spark.read.table("bronze_products")
df_custs  = spark.read.table("bronze_customers")

# Without broadcast
t0 = time.time()
result_normal = (df_orders
    .join(df_prods, "product_id")
    .join(df_custs, "customer_id")
    .groupBy("category","segment")
    .agg(spark_count("order_id").alias("orders"))
)
result_normal.show(5)
t_normal = time.time() - t0
print(f"Without broadcast hint: {t_normal:.2f}s")


In [ ]:
# ── Step 11e.3  Join WITH broadcast hint
# broadcast(df) tells Spark: "trust me, this is small - send it to all executors"

t0 = time.time()
result_broadcast = (df_orders
    .join(broadcast(df_prods), "product_id")   # ← broadcast the small products table
    .join(broadcast(df_custs), "customer_id")  # ← broadcast the small customers table
    .groupBy("category","segment")
    .agg(spark_count("order_id").alias("orders"))
)
result_broadcast.show(5)
t_broadcast = time.time() - t0
print(f"With broadcast hint   : {t_broadcast:.2f}s")


In [ ]:
# ── Step 11e.4  Verify Spark chose a BroadcastHashJoin (not SortMergeJoin)
#
# .explain() shows the physical plan - look for BroadcastHashJoin vs SortMergeJoin

print("Physical plan (look for BroadcastHashJoin):")
result_broadcast.explain(mode="formatted")


In [ ]:
# ── Step 11e.5  Set the auto-broadcast threshold (optional)
#
# Increase if your dimension tables are larger than the 10 MB default

# Check current setting
current = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"Current autoBroadcastJoinThreshold: {int(current)/1024/1024:.0f} MB")

# Set to 100 MB (common production setting for small-ish dimension tables)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)
print("Updated autoBroadcastJoinThreshold to 100 MB ✓")


---
## 🏁 Summary - What We Built

| Section | Transformation | PySpark |
|---------|---------------|-----------------|
| 1 | Split column by delimiter | `split()`, `getItem()` |
| 2 | Extract before/after delimiter | `split().getItem(0/1)`, `regexp_extract()` |
| 3 | Replace values | `when/otherwise`, `regexp_replace()`, `.replace()` |
| 4 | Merge queries (JOIN) | `.join()`, `broadcast()` |
| 5 | Append queries (UNION) | `unionByName()`, `row_number()` window dedup |
| 6 | Pivot | `.pivot()` + `.agg()` |
| 7 | Unpivot | `stack()` via `expr()` |
| 8 | Uppercase / Lowercase | `upper()`, `lower()`, `initcap()` |
| 9 | Trim whitespace | `trim()`, `ltrim()`, `rtrim()`, `regexp_replace \s+` |
| 10 | Incremental loading | Watermark + append, `DeltaTable.merge()` |
| 11 | Engine optimisations | V-Order, OPTIMIZE, VACUUM, `.cache()`, `broadcast()` |

### The 3 things to take home

1. **Every M step has a PySpark equivalent** - but the PySpark version is a reusable function
2. **Dataflows Gen2 incremental refresh has real limits** - bucket replace, folding required,
   OPTIMIZE blocked. Delta MERGE gives you row-level control with no restrictions.
3. **Always run `OPTIMIZE … VORDER` after a significant load** - it's one SQL line and makes
   Power BI Direct Lake 40–60% faster on cold-cache queries.

---
 - **Thomas Leblanc** | FabCon Atlanta 2026
- **Nikola Ilic** | data-mozart.com | FabCon Atlanta 2026
